# Nutrient validation — GETM–BFM vs NIOZ Jetty and RWS chemistry

Compares surface nutrients, chlorophyll and oxygen of one or more model runs
(default `spinup_10`) with two observation sets in
`/export/lv9/projects/dws/results/validation/nutrients/`:

| source | file | nature |
|---|---|---|
| **NIOZ Jetty** | `Jetty_HWseries.csv` | one fixed station (Marsdiep, Texel), samples at high water |
| **RWS** | `Chemistry_data_via_Waterinfo_RWS.csv` | monitoring stations, irregular bottle samples |

**Model variables** — `N1p` (phosphate), `N3n` (nitrate), `N4n` (ammonium),
`N5s` (silicate), `Chla`, `O2o`, and the derived `DIN = N3n + N4n` and DIN:DIP.
BFM nutrients are in mmol m⁻³ (= µmol L⁻¹), `Chla` in mg m⁻³ (= µg L⁻¹).

**Sampling** — surface layer (`level = 10`; top = 10, bottom = 0), nearest *wet*
model cell (distance in km), and the model day that contains the sample.

### Read before interpreting

* `spinup_10` is a **repeated 2015 year**, so only 2015 observations are used:
  this tests the seasonal cycle and the spatial gradient, not inter-annual skill.
* In `spinup_10` the BFM fields are **daily means** (run `README.txt`, and the
  `averaged` attribute on every variable): the value stamped at 00:00 is the mean
  of the preceding day. Each bottle sample is therefore paired with the mean of
  the day it was taken on, not with the nearest midnight. With
  `MODEL_TIME_CONVENTION = "auto"` this is read from the file, so runs with
  instantaneous output (spinup_01–09) are matched to the nearest time instead.
* The Jetty samples are taken at **high water**, when North Sea water fills the
  inlet; a daily mean mixes both tidal phases, so part of any offset there is a
  sampling effect rather than model error.
* Model `Chla` is a diagnostic sum over the phytoplankton groups; observed
  chlorophyll methods differ between the two records.
* In 2015 the RWS export holds three stations (Marsdiep Noord, Vliestroom,
  Terschelling 10 km) without silicate, and the Jetty series has no oxygen.

### Figures

In `OUT_DIR/figures/`, each as vector PDF + 600 dpi PNG + CSV of the plotted
numbers, in the shared house style (`figstyle.py`):

| File | Content |
|---|---|
| `fig01_nutrients_stations` | stations, model domain and bathymetry |
| `fig02_nutrients_timeseries` | seasonal cycle of every variable at every station |
| `fig03_nutrients_scatter` | model vs observed, all stations pooled |
| `fig04_nutrients_scorecard_<run>` | normalised bias and correlation per variable and station |
| `fig05_nutrients_stoichiometry` | DIN:DIP against the Redfield ratio |

---
## 1. Configuration

The only cell you should normally need to edit.

In [ ]:
import csv
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from scipy.spatial import cKDTree
from scipy.stats import pearsonr

try:                                   # only needed for projected RWS coordinates
    from pyproj import Transformer
    _HAS_PYPROJ = True
except Exception:
    _HAS_PYPROJ = False

sys.path.insert(0, str(Path.cwd()))    # figstyle.py sits next to this notebook
import figstyle as fs

fs.use_style()

# --- paths -------------------------------------------------------------------
POSTPROC_DIR    = Path("/export/lv9/projects/dws/results/validation/nutrients/")
BASE_OUTPUT_DIR = Path("/export/lv9/projects/dws/model_output/archived_runs/")
SPINUP_NAMES    = ["spinup_10"]
OUT_DIR = POSTPROC_DIR / SPINUP_NAMES[0]      # collocated tables, metrics
FIG_DIR = OUT_DIR / "figures"                 # figures + their CSV twins
OBS_CSV = {
    "NIOZ_JETTY": POSTPROC_DIR / "Jetty_HWseries.csv",
    "RWS":        POSTPROC_DIR / "Chemistry_data_via_Waterinfo_RWS.csv",
}
NC_PATTERN = "dws_500m.3d.{year}*.nc"
PROBE_SCHEMA = False       # True: print the full layout of both CSVs (new exports)

# --- model sampling ----------------------------------------------------------
MODEL_YEAR = 2015          # spinup runs repeat 2015
SURFACE_LEVEL = 10         # level has 11 layers, top = 10
# "auto": read the `averaged` attribute; "interval_mean": a value stamped t is
# the mean over the preceding interval; "instantaneous": nearest time stamp.
MODEL_TIME_CONVENTION = "auto"
BOX_HALF = 1               # (2*BOX_HALF+1)^2 wet cells around a station -> model band
JETTY_LON, JETTY_LAT = 4.7891, 53.0018        # NIOZ jetty, Marsdiep (Texel)

# --- variables -----------------------------------------------------------------
VALIDATION_VARS = ["N1p", "N3n", "N4n", "N5s", "Chla", "O2o"]
DERIVED_VARS = ["DIN", "DIN_DIP"]
ALL_VARS = VALIDATION_VARS + DERIVED_VARS
PLOT_VARS = ["N3n", "N4n", "N1p", "N5s", "Chla", "O2o"]     # figure row order
VAR_LABEL = {"N1p": "Phosphate", "N3n": "Nitrate", "N4n": "Ammonium", "N5s": "Silicate",
             "Chla": "Chlorophyll a", "O2o": "Oxygen", "DIN": "DIN", "DIN_DIP": "DIN:DIP"}
VAR_UNITS = {"N1p": r"mmol P m$^{-3}$", "N3n": r"mmol N m$^{-3}$", "N4n": r"mmol N m$^{-3}$",
             "N5s": r"mmol Si m$^{-3}$", "Chla": r"mg m$^{-3}$", "O2o": r"mmol O$_2$ m$^{-3}$",
             "DIN": r"mmol N m$^{-3}$", "DIN_DIP": r"mol mol$^{-1}$"}
# Rough order-of-magnitude ranges (BFM plotting script, J. van der Molen);
# used only in the unit audit, never to filter data.
PLAUSIBLE_RANGE = {"N1p": (0, 6), "N3n": (0, 150), "N4n": (0, 15), "N5s": (0, 100),
                   "O2o": (100, 450), "Chla": (0, 100), "DIN": (0, 165), "DIN_DIP": (0, 200)}
REDFIELD_NP = 16.0

# --- grid and observation handling ----------------------------------------------
LON_VALID, LAT_VALID = (-180.0, 180.0), (-90.0, 90.0)    # lonc/latc fill is -999
BATHY_FILL = -10.0                  # land cells carry bathymetry == -10
DATA_FILL_THRESHOLD = -900.0        # -9999 fill and -9998 empty first frame -> NaN
RWS_MISSING_SENTINEL = 999999999.0  # DONAR "no value" code
DROP_BELOW_DETECTION = False        # False: keep as LOD/2 (drawn hollow); True: drop
MAX_SNAP_DISTANCE_KM = 5.0          # stations further from a wet cell are dropped
MAX_TIME_OFFSET_DAYS = 3.0          # pairs further apart in time are voided
MIN_PAIRS = 5                       # fewer pairs: no skill score

# --- figures ---------------------------------------------------------------------
STATION_LABEL = {"NIOZ_JETTY": "NIOZ Jetty", "MARSDND": "Marsdiep N",
                 "VLIESM": "Vliestroom", "TERSLG10": "Terschelling 10 km"}
STATION_ORDER = ["NIOZ_JETTY", "MARSDND", "VLIESM", "TERSLG10"]   # others follow by longitude
MAP_EXTENT = (4.5, 5.6, 52.88, 53.52)
RUN_COLOURS = fs.run_colours(SPINUP_NAMES)

FIG_DIR.mkdir(parents=True, exist_ok=True)
print("model runs:", ", ".join(f"{s} ({'found' if (BASE_OUTPUT_DIR / s).is_dir() else 'MISSING'})"
                                for s in SPINUP_NAMES))
print("figures to:", FIG_DIR)

---
## 2. Schema probe (optional)

Prints the layout of both CSVs — separator, columns and, for the RWS export,
every parameter, unit and `hoedanigheid` — which is what the unit conversion in
§3 is keyed on. Set `PROBE_SCHEMA = True` when a new export arrives.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# SCHEMA PROBE  —  run this first
# ──────────────────────────────────────────────────────────────────────────────
# No existing script in this repo reads Chemistry_data_via_Waterinfo_RWS.csv, so
# its layout is confirmed here rather than assumed.  Read the printout before
# trusting anything downstream: the parameter / eenheid / hoedanigheid
# combinations listed below are exactly what the unit conversion is keyed on.

_DESCRIPTOR_HINTS = (
    "parameter", "grootheid", "eenheid", "hoedanigheid", "locatie", "limiet",
    "kwaliteit", "compartiment", "meetpunt", "waardebepaling",
)


def sniff_delimiter(path: Path) -> str:
    """Guess the field separator from the header line."""
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        sample = fh.readline() + fh.readline()
    try:
        return csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
    except csv.Error:
        # Fall back on whichever candidate appears most often in the header.
        counts = {d: sample.count(d) for d in [";", ",", "\t", "|"]}
        return max(counts, key=counts.get)


def probe_csv(name: str, path: Path, n_preview: int = 5) -> pd.DataFrame:
    """Print the layout of one observation CSV and return it as raw strings."""
    print("=" * 78)
    print(f"{name}  —  {path}")
    print("=" * 78)
    if not path.exists():
        print("  [ERROR] file not found")
        return pd.DataFrame()

    sep = sniff_delimiter(path)
    print(f"detected separator: {sep!r}")

    # dtype=str keeps Dutch decimal commas and leading zeros intact; every
    # numeric column is cast explicitly further down.
    df = pd.read_csv(path, sep=sep, dtype=str, na_values=["NA", ""],
                     encoding="utf-8", encoding_errors="replace",
                     engine="python")
    print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print("\ncolumns:")
    for c in df.columns:
        n_uniq = df[c].nunique(dropna=True)
        example = df[c].dropna().iloc[0] if df[c].notna().any() else "—"
        print(f"  {c:<38} nunique={n_uniq:<8} e.g. {str(example)[:40]}")

    print("\nfirst rows:")
    with pd.option_context("display.max_columns", None, "display.width", 250):
        print(df.head(n_preview))

    # Descriptor columns: these drive the parameter/unit mapping.
    desc_cols = [c for c in df.columns
                 if any(h in c.lower() for h in _DESCRIPTOR_HINTS)]
    for c in desc_cols:
        vc = df[c].value_counts(dropna=False)
        if len(vc) > 60:
            print(f"\n{c}: {len(vc)} distinct values (showing top 30)")
            print(vc.head(30).to_string())
        else:
            print(f"\n{c}:")
            print(vc.to_string())

    # Anything that could be a coordinate — the magnitude tells us the CRS.
    for c in df.columns:
        if c.lower().split(".")[-1] in ("x", "y", "lon", "lat", "longitude", "latitude"):
            vals = pd.to_numeric(df[c].str.replace(",", ".", regex=False), errors="coerce")
            if vals.notna().any():
                print(f"\ncoordinate column {c}: min={vals.min():.4f} max={vals.max():.4f}")

    return df


if PROBE_SCHEMA:
    _probe_frames = {name: probe_csv(name, path) for name, path in OBS_CSV.items()}
else:
    print("schema probe skipped (PROBE_SCHEMA = False)")

---
## 3. Units

Target units: **mmol m⁻³** for the molar species (identical to µmol L⁻¹) and
**mg m⁻³** for chlorophyll (identical to µg L⁻¹). RWS reports the same substance
both as element mass and as whole-ion mass, which differ by a large factor:

| reported as | → mmol m⁻³ per mg L⁻¹ | ratio to element basis |
|---|---|---|
| mg **N** L⁻¹ | 71.39 | — |
| mg **NO₃** L⁻¹ | 16.13 | 4.43 |
| mg **P** L⁻¹ | 32.29 | — |
| mg **PO₄** L⁻¹ | 10.53 | 3.07 |
| mg **Si** L⁻¹ | 35.61 | — |
| mg **SiO₂** L⁻¹ | 16.64 | 2.14 |

`unit_factor()` resolves the basis from `hoedanigheid` when it names an element,
falls back on the ion implied by the parameter code, and raises on anything
else rather than defaulting to 1.0. The Jetty nutrients are µmol L⁻¹ and `Chl`
mg m⁻³ (NIOZ Dataverse, doi:10.25850/nioz/7b.b.5j), so they need no scaling; the
unit audit in §6 re-checks that against the model.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# UNIT CONVERSION
# ──────────────────────────────────────────────────────────────────────────────
# Target units:  molar species -> mmol m-3  (identical to umol L-1)
#                Chla          -> mg m-3    (identical to ug L-1)
#
# RWS reports the same substance both as element mass and as compound mass.
# mg NO3/L and mg N/L differ by a factor 4.43, mg PO4/L and mg P/L by 3.07;
# picking the wrong one produces figures that look plausible and are wrong.
# The conversion is therefore keyed on the (parameter, hoedanigheid, eenheid)
# triplet and *raises* on anything it does not recognise.

MOLAR_MASS = {
    # elements / molecules used as the mass basis
    "N": 14.0067, "P": 30.973762, "Si": 28.0855, "O2": 31.9988, "C": 12.011,
    # whole ions, used when hoedanigheid does not name an element
    "NO3": 62.0049, "NO2": 46.0055, "NH4": 18.0385,
    "PO4": 94.9714, "SiO2": 60.0843,
}

# Which whole molecule a parameter code refers to, used only when the
# hoedanigheid field does not already state the mass basis (N / P / Si / O2).
PARAM_COMPOUND = {
    "NO3": "NO3", "NO2": "NO2", "NH4": "NH4",
    "NO3NO2": "NO3", "NO2NO3": "NO3",   # combined NOx, mass-basis fallback
    "PO4": "PO4", "SIO2": "SiO2", "SI": "SiO2", "O2": "O2",
}

# hoedanigheid_code variants seen in the RWS Waterinfo export (Nnf/Npg/Pnf/...
# = dissolved/particulate-bound fractions expressed as the given element).
HOED_BASIS = {
    "N": "N", "NNF": "N", "NPG": "N",
    "P": "P", "PNF": "P", "PG": "P",
    "SI": "Si", "SINF": "Si",
    "O2": "O2",
    "C": "C", "CNF": "C", "CPG": "C", "CDG": "C",
}

# Volume conversion to grams per litre.
_MASS_PER_LITRE = {
    "g/l": 1.0, "mg/l": 1e-3, "ug/l": 1e-6, "ng/l": 1e-9,
    "g/m3": 1e-3, "mg/m3": 1e-6, "ug/m3": 1e-9,
    "mg/dm3": 1e-3, "ug/dm3": 1e-6,
}
# Already-molar units, expressed as a factor to mmol m-3.
_MOLAR_TO_MMOL_M3 = {
    "umol/l": 1.0, "umol/dm3": 1.0, "mmol/m3": 1.0, "umol/kg": 1.0,
    "mmol/l": 1e3, "mol/m3": 1e3, "mmol/dm3": 1e3,
    "nmol/l": 1e-3, "umol/m3": 1e-3,
}
# Chlorophyll is a mass concentration, not molar.
_CHL_TO_MG_M3 = {
    "ug/l": 1.0, "mg/m3": 1.0, "ug/dm3": 1.0,
    "mg/l": 1e3, "g/m3": 1e3, "ng/l": 1e-3, "ug/m3": 1e-3,
}

# RWS parameter code -> model variable.  Codes are upper-cased and stripped of
# separators before lookup, so "NO3-N", "no3_n" and "NO3" all match "NO3N"/"NO3".
# EXTEND THIS after reading the schema-probe printout.
RWS_PARAM_MAP = {
    "NO3": "N3n", "NO3N": "N3n", "NITRAAT": "N3n",
    "NO3NO2": "N3n", "NO2NO3": "N3n",   # combined NOx reported as nitrate
    "NH4": "N4n", "NH4N": "N4n", "AMMONIUM": "N4n",
    "PO4": "N1p", "PO4P": "N1p", "FOSFAAT": "N1p", "OPGELOSTORTHOFOSFAAT": "N1p",
    "SIO2": "N5s", "SI": "N5s", "SILICAAT": "N5s",
    "O2": "O2o", "ZUURSTOF": "O2o",
    "CHLFA": "Chla", "CHLFAA": "Chla", "CHLA": "Chla", "CHLOROFYLA": "Chla",
}
# Deliberately NOT mapped – total nutrient pools are not comparable to the
# model's dissolved inorganic pools:  Ntot, Ptot, Kj (Kjeldahl), NO2 alone.
RWS_PARAM_IGNORE = {"NTOT", "PTOT", "NKJ", "KJN", "NO2", "PO4TOT", "NTOTAAL", "PTOTAAL"}


def _norm_code(value) -> str:
    """Upper-case a code and strip spaces, dashes, underscores and dots."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    return re.sub(r"[\s\-_.]", "", str(value)).upper()


def _norm_unit(value) -> str:
    """Normalise a unit string: lower case, ASCII mu, no spaces, no superscripts."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    u = str(value).strip().lower()
    u = u.replace("µ", "u").replace("μ", "u")   # micro sign / greek mu
    u = u.replace("³", "3").replace("^3", "3").replace("**3", "3")
    u = u.replace(" ", "")
    # "mg/l N" style: the mass basis is handled separately, drop the trailing tag
    u = re.sub(r"(n|p|si|o2|c)$", "", u) if re.match(r"^[a-z]+/[a-z0-9]+(n|p|si|o2|c)$", u) else u
    return u


def unit_factor(model_var: str, param_code: str, hoedanigheid: str, eenheid: str) -> float:
    """
    Factor by which an observed value must be multiplied to reach the model unit
    (mmol m-3, or mg m-3 for Chla).

    Raises ValueError on any combination that is not explicitly covered — a
    silent fallback to 1.0 here is the most dangerous failure mode in the whole
    notebook, so it is not offered.
    """
    unit = _norm_unit(eenheid)
    hoed = _norm_code(hoedanigheid)
    param = _norm_code(param_code)

    if model_var == "Chla":
        if unit in _CHL_TO_MG_M3:
            return _CHL_TO_MG_M3[unit]
        raise ValueError(
            f"Unrecognised chlorophyll unit {eenheid!r} (normalised {unit!r}). "
            f"Add it to _CHL_TO_MG_M3."
        )

    if unit in _MOLAR_TO_MMOL_M3:
        return _MOLAR_TO_MMOL_M3[unit]

    if unit not in _MASS_PER_LITRE:
        raise ValueError(
            f"Unrecognised unit {eenheid!r} (normalised {unit!r}) for parameter "
            f"{param_code!r}. Add it to _MASS_PER_LITRE or _MOLAR_TO_MMOL_M3."
        )

    # Mass basis: hoedanigheid wins when it names an element, otherwise the
    # value is the mass of the whole ion implied by the parameter code.
    basis = HOED_BASIS.get(hoed)
    if basis is None:
        basis = PARAM_COMPOUND.get(param)
    if basis is None:
        raise ValueError(
            f"Cannot determine the mass basis for parameter {param_code!r} with "
            f"hoedanigheid {hoedanigheid!r} and unit {eenheid!r}. Either add the "
            f"parameter to PARAM_COMPOUND or map the hoedanigheid code."
        )

    # value [unit] -> g/L -> mol/L -> umol/L == mmol/m3
    return _MASS_PER_LITRE[unit] / MOLAR_MASS[basis] * 1e6


# ── NIOZ Jetty ────────────────────────────────────────────────────────────────
# Metadata documented in output_scripts/pelagic_validation.ipynb (cell 2) and in
# the NIOZ Dataverse series (doi:10.25850/nioz/7b.b.5j):
#   TSM mg/L | Chl mg/m3 | TOC, POC, DOC mgC/L | NO3, NH4, PO4 umol/L
# umol/L is numerically identical to mmol/m3, so the nutrients need no scaling.
# The unit-audit cell below re-checks this against the model percentiles, which
# is what would expose an unexpected unit as an order-of-magnitude offset.
JETTY_COL_MAP = {
    "Chl": "Chla",
    "NO3": "N3n",
    "NH4": "N4n",
    "PO4": "N1p",
    "SiO2": "N5s",   # present in some releases of the series; skipped if absent
    "Si": "N5s",
    "O2": "O2o",
}
JETTY_FACTOR = {
    "Chla": 1.0,   # mg m-3   -> mg m-3
    "N3n": 1.0,    # umol L-1 -> mmol m-3
    "N4n": 1.0,
    "N1p": 1.0,
    "N5s": 1.0,
    "O2o": 1.0,
}

print("Unit conversion self-test (value 1.0 -> mmol m-3):")
for _p, _h, _u in [("NO3", "N", "mg/l"), ("NO3", "NVT", "mg/l"),
                   ("PO4", "P", "mg/l"), ("PO4", "NVT", "mg/l"),
                   ("NH4", "N", "ug/l"), ("SiO2", "Si", "mg/l"),
                   ("O2", "NVT", "mg/l"), ("NO3", "N", "umol/l")]:
    _mv = RWS_PARAM_MAP[_norm_code(_p)]
    print(f"  {_p:<5} hoedanigheid={_h:<4} {_u:<7} -> x {unit_factor(_mv, _p, _h, _u):9.4f}  ({_mv})")
print(f"  CHLFA hoedanigheid=NVT  ug/l    -> x {unit_factor('Chla', 'CHLFA', 'NVT', 'ug/l'):9.4f}  (Chla)")

---
## 4. Observation loaders

`load_rws()` auto-detects which of two RWS export schemas the file uses, reads
everything as strings (Dutch decimal commas survive), drops the DONAR
`999999999` missing code, keeps `<` detection-limit values as LOD/2 with a
`below_lod` flag (drawn hollow), keeps surface water only, and converts station
coordinates from RD or UTM when needed. `load_jetty()` reshapes the fixed-station
series into the same layout.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# OBSERVATION LOADERS
# ──────────────────────────────────────────────────────────────────────────────

# Canonical name -> candidate column names, lower-cased.  Two RWS export
# schemas are in circulation and both are handled:
#   * DONAR/Waterinfo download  : ';'-separated, UPPERCASE, LAT/LON, dd-mm-yyyy
#     (the schema parsed in pelagic_validation.ipynb cell 7)
#   * Waterinfo API / ddlpy     : ','-separated, dotted lower case, ISO tijdstip
#     (the schema parsed in pelagic_validation.ipynb cell 13)
RWS_COL_CANDIDATES = {
    "station":   ["locatie_code", "locatie.code", "locatiecode", "meetpunt.identificatie",
                  "locatie.naam", "locatie_naam", "meetpuntidentificatie",
                  "code", "naam"],
    "datetime":  ["tijdstip", "datumtijd", "datum_tijd"],
    "date":      ["waarnemingdatum", "datum", "waarneming_datum"],
    "time":      ["waarnemingtijd", "tijd", "waarneming_tijd"],
    "value":     ["numeriekewaarde", "waarde", "meetwaarde"],
    "parameter": ["parameter_code", "parameter.code", "parameter"],
    "parameter_desc": ["parameter_wat_omschrijving", "parameter_omschrijving"],
    "quantity":  ["grootheid_code", "grootheid.code", "grootheid"],
    "unit":      ["eenheid_code", "eenheid.code", "eenheid"],
    "basis":     ["hoedanigheid_code", "hoedanigheid.code", "hoedanigheid"],
    "limit":     ["limietsymbool", "limiet_symbool", "limietsymbool.code"],
    "lon":       ["lon", "longitude", "geografischepunt.x"],
    "lat":       ["lat", "latitude", "geografischepunt.y"],
    "x":         ["x", "locatie.x", "geometriepunt.x", "xcoordinaat"],
    "y":         ["y", "locatie.y", "geometriepunt.y", "ycoordinaat"],
    "geom":      ["geom", "geometrie", "wkt"],
    "compartment": ["compartiment_code", "compartiment.code", "compartiment"],
}


def _resolve_columns(df: pd.DataFrame) -> dict:
    """Map canonical names onto the columns actually present in *df*."""
    lower = {c.lower().strip(): c for c in df.columns}
    found = {}
    for canon, candidates in RWS_COL_CANDIDATES.items():
        for cand in candidates:
            if cand in lower:
                found[canon] = lower[cand]
                break
    return found


def _to_float(series: pd.Series) -> pd.Series:
    """Cast a string column to float, tolerating Dutch decimal commas."""
    return pd.to_numeric(
        series.astype(str).str.strip().str.replace(",", ".", regex=False),
        errors="coerce",
    )


def _classify_rws_param(desc: str) -> str:
    """
    Map a Dutch parameter_wat_omschrijving string onto a RWS_PARAM_MAP key.

    Needed because this export's parameter_code/grootheid_code columns are
    always "NVT" — the substance name only appears in this free-text field.
    Returns "" for anything not relevant to validation (e.g. totals, Kjeldahl).
    """
    d = str(desc).lower()
    if "ammonium" in d:
        return "NH4"
    if "som nitraat en nitriet" in d:
        return "NO3NO2"
    if "nitraat" in d and "nitriet" not in d:
        return "NO3"
    if "nitriet" in d and "nitraat" not in d:
        return "NO2"
    if "fosfaat" in d:
        return "PO4"
    if "silicaat" in d or "silicium" in d:
        return "SIO2"
    if "chlorofyl" in d:
        return "CHLFA"
    if "zuurstof" in d:
        return "O2"
    return ""


def parse_geom(geom_str):
    """Parse 'POINT (lon lat)' -> (lon, lat).  From pelagic_validation.ipynb."""
    if isinstance(geom_str, str) and geom_str.upper().startswith("POINT"):
        coords = geom_str[geom_str.index("(") + 1: geom_str.index(")")].split()
        return float(coords[0]), float(coords[1])
    return np.nan, np.nan


def to_wgs84(x: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray, str]:
    """
    Convert projected station coordinates to lon/lat, detecting the CRS from the
    magnitude of the values.  No precedent exists in this repo, so the detected
    CRS is printed rather than assumed silently.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    xm, ym = np.nanmax(np.abs(x)), np.nanmax(np.abs(y))

    if xm <= 180.0 and ym <= 90.0:
        return x, y, "EPSG:4326 (already lon/lat)"
    if xm < 3.5e5 and 3.0e5 < ym < 7.0e5:
        src = "EPSG:28992"          # Rijksdriehoek
    elif 3.0e5 < xm < 9.0e5 and ym > 5.0e6:
        src = "EPSG:25831"          # UTM zone 31N, what ddlpy returns
    else:
        raise ValueError(
            f"Cannot identify the CRS of the station coordinates "
            f"(max |x| = {xm:.1f}, max |y| = {ym:.1f}). Set it explicitly."
        )
    if not _HAS_PYPROJ:
        raise RuntimeError(f"Coordinates look like {src} but pyproj is unavailable.")
    tf = Transformer.from_crs(src, "EPSG:4326", always_xy=True)
    lon, lat = tf.transform(x, y)
    return np.asarray(lon), np.asarray(lat), src


def load_rws() -> pd.DataFrame:
    """
    Read the RWS chemistry export and return one row per (station, timestamp,
    model_var) with the value converted to the model's unit.

    Columns: station, lon, lat, timestamp, model_var, obs_value, below_lod.
    """
    path = OBS_CSV["RWS"]
    sep = sniff_delimiter(path)
    raw = pd.read_csv(path, sep=sep, dtype=str, na_values=["NA", ""],
                      encoding="utf-8", encoding_errors="replace", engine="python")
    cols = _resolve_columns(raw)
    print(f"RWS: {len(raw):,} raw rows, separator {sep!r}")
    print(f"     resolved columns: {cols}")

    missing = [k for k in ("station", "value", "unit") if k not in cols]
    if missing:
        raise KeyError(
            f"Could not find RWS column(s) {missing} in {list(raw.columns)}. "
            f"Extend RWS_COL_CANDIDATES."
        )

    df = pd.DataFrame(index=raw.index)
    df["station"] = raw[cols["station"]].astype(str).str.strip()

    # ── timestamp ─────────────────────────────────────────────────────────────
    if "datetime" in cols:
        ts = pd.to_datetime(raw[cols["datetime"]], errors="coerce", utc=True)
        df["timestamp"] = ts.dt.tz_convert(None)
    elif "date" in cols:
        combined = raw[cols["date"]].astype(str)
        if "time" in cols:
            combined = combined + " " + raw[cols["time"]].astype(str)
        df["timestamp"] = pd.to_datetime(combined, dayfirst=True, errors="coerce")
    else:
        raise KeyError("No date/time column found in the RWS export.")

    # ── substance, unit, mass basis ───────────────────────────────────────────
    # parameter.code names the substance; grootheid.code names the quantity
    # (CONCTTE etc.).  Prefer the parameter, fall back on the quantity.
    # Some RWS exports leave parameter_code == "NVT" for every row; the
    # substance then only appears in the free-text parameter_wat_omschrijving.
    if "parameter_desc" in cols:
        df["param_raw"] = raw[cols["parameter_desc"]].map(_classify_rws_param)
    else:
        param_col = cols.get("parameter") or cols.get("quantity")
        df["param_raw"] = raw[param_col].astype(str).str.strip()
    df["unit_raw"] = raw[cols["unit"]].astype(str).str.strip()
    df["basis_raw"] = raw[cols["basis"]].astype(str).str.strip() if "basis" in cols else ""

    # ── value, missing codes, detection limits ────────────────────────────────
    df["raw_value"] = _to_float(raw[cols["value"]])
    n0 = len(df)
    df = df[np.isfinite(df["raw_value"])]
    df = df[df["raw_value"] != RWS_MISSING_SENTINEL]
    df = df[df["raw_value"] < RWS_MISSING_SENTINEL / 1e3]   # any 9.99e8-style code
    print(f"     dropped {n0 - len(df):,} rows with missing / sentinel values")

    if "limit" in cols:
        lim = raw.loc[df.index, cols["limit"]].astype(str).str.strip()
        df["below_lod"] = lim.eq("<")
    else:
        df["below_lod"] = False
    n_lod = int(df["below_lod"].sum())
    if n_lod:
        if DROP_BELOW_DETECTION:
            df = df[~df["below_lod"]]
            print(f"     dropped {n_lod:,} below-detection-limit rows")
        else:
            df.loc[df["below_lod"], "raw_value"] *= 0.5
            print(f"     {n_lod:,} below-detection-limit rows substituted with LOD/2")

    # ── surface water only, when the export says which compartment ────────────
    if "compartment" in cols:
        comp = raw.loc[df.index, cols["compartment"]].astype(str).str.strip().str.upper()
        keep = comp.isin(["OW", "OPPERVLAKTEWATER", ""]) | comp.isna()
        if (~keep).any():
            print(f"     dropped {int((~keep).sum()):,} rows outside compartment OW")
            df = df[keep]

    # ── map parameters onto model variables ───────────────────────────────────
    codes = df["param_raw"].map(_norm_code)
    df["model_var"] = codes.map(RWS_PARAM_MAP)
    unmapped = sorted(set(codes[df["model_var"].isna()]) - RWS_PARAM_IGNORE - {""})
    if unmapped:
        print(f"     [INFO] parameters present but not mapped: {unmapped}")
        print(f"            add them to RWS_PARAM_MAP if they should be validated")
    df = df[df["model_var"].notna()].copy()

    # ── unit conversion ───────────────────────────────────────────────────────
    combos = df[["model_var", "param_raw", "basis_raw", "unit_raw"]].drop_duplicates()
    factors = {}
    print("     unit conversions applied:")
    for _, r in combos.iterrows():
        key = (r["model_var"], r["param_raw"], r["basis_raw"], r["unit_raw"])
        factors[key] = unit_factor(r["model_var"], r["param_raw"], r["basis_raw"], r["unit_raw"])
        print(f"       {r['param_raw']:<10} [{r['unit_raw']:<8}] hoedanigheid="
              f"{r['basis_raw'] or '-':<5} -> {r['model_var']:<5} x {factors[key]:.5g}")

    keys = list(zip(df["model_var"], df["param_raw"], df["basis_raw"], df["unit_raw"]))
    df["obs_value"] = df["raw_value"].to_numpy() * np.array([factors[k] for k in keys])

    # ── station coordinates ───────────────────────────────────────────────────
    if "lon" in cols and "lat" in cols:
        lon = _to_float(raw.loc[df.index, cols["lon"]])
        lat = _to_float(raw.loc[df.index, cols["lat"]])
        df["lon"], df["lat"] = lon.to_numpy(), lat.to_numpy()
        print("     coordinates: LAT/LON columns, EPSG:4326")
    elif "geom" in cols:
        xy = raw.loc[df.index, cols["geom"]].map(parse_geom)
        df["lon"] = [p[0] for p in xy]
        df["lat"] = [p[1] for p in xy]
        print("     coordinates: parsed from WKT geom, EPSG:4326")
    elif "x" in cols and "y" in cols:
        lon, lat, src = to_wgs84(_to_float(raw.loc[df.index, cols["x"]]).to_numpy(),
                                 _to_float(raw.loc[df.index, cols["y"]]).to_numpy())
        df["lon"], df["lat"] = lon, lat
        print(f"     coordinates: x/y columns interpreted as {src}")
    else:
        raise KeyError("No station coordinates found in the RWS export.")

    df = df.dropna(subset=["timestamp", "lon", "lat", "obs_value"])
    out = df[["station", "lon", "lat", "timestamp", "model_var", "obs_value", "below_lod"]]
    out = out.sort_values(["station", "timestamp"]).reset_index(drop=True)
    print(f"     {len(out):,} usable rows, "
          f"{out['station'].nunique()} stations, "
          f"{out['timestamp'].dt.year.min()}–{out['timestamp'].dt.year.max()}")
    return out


def rws_to_wide(obs_long: pd.DataFrame) -> pd.DataFrame:
    """
    One row per water sample (station + timestamp), one column per model
    variable.  DIN and DIN:DIP are only defined where both species were
    measured in the same sample.
    """
    keys = ["station", "lon", "lat", "timestamp"]
    wide = (obs_long
            .pivot_table(index=keys, columns="model_var",
                         values="obs_value", aggfunc="mean")
            .reset_index())
    wide.columns.name = None

    lod = (obs_long
           .pivot_table(index=keys, columns="model_var",
                        values="below_lod", aggfunc="max")
           .reset_index())
    lod.columns.name = None
    lod = lod.rename(columns={v: f"{v}_below_lod" for v in obs_long["model_var"].unique()})
    wide = wide.merge(lod, on=keys, how="left")

    for v in VALIDATION_VARS:
        if v not in wide.columns:
            wide[v] = np.nan
        flag = f"{v}_below_lod"
        wide[flag] = wide[flag].fillna(False).astype(bool) if flag in wide.columns else False
    return add_derived(wide)


def add_derived(df: pd.DataFrame, suffix: str = "") -> pd.DataFrame:
    """Add DIN and DIN:DIP to a table holding {N3n,N4n,N1p}{suffix} columns."""
    n3, n4, n1 = f"N3n{suffix}", f"N4n{suffix}", f"N1p{suffix}"
    if n3 in df.columns and n4 in df.columns:
        df[f"DIN{suffix}"] = df[n3] + df[n4]
    else:
        df[f"DIN{suffix}"] = np.nan
    if f"DIN{suffix}" in df.columns and n1 in df.columns:
        dip = df[n1].where(df[n1] > 0.01)      # guard division by ~zero phosphate
        df[f"DIN_DIP{suffix}"] = df[f"DIN{suffix}"] / dip
    else:
        df[f"DIN_DIP{suffix}"] = np.nan
    return df


def load_jetty() -> pd.DataFrame:
    """
    NIOZ jetty high-water series -> one row per sampling time, one column per
    model variable, in model units.  Follows pelagic_validation.ipynb cell 24.
    """
    path = OBS_CSV["NIOZ_JETTY"]
    raw = pd.read_csv(path, na_values=["NA", ""], parse_dates=["timestamp"])
    print(f"Jetty: {len(raw):,} raw rows, columns {list(raw.columns)}")

    raw = raw.dropna(subset=["timestamp"])
    out = pd.DataFrame({"timestamp": raw["timestamp"]})
    for obs_col, mvar in JETTY_COL_MAP.items():
        if obs_col not in raw.columns:
            continue
        if mvar in out.columns and out[mvar].notna().any():
            continue        # first matching alias wins (e.g. SiO2 before Si)
        out[mvar] = pd.to_numeric(raw[obs_col], errors="coerce") * JETTY_FACTOR[mvar]
        print(f"       {obs_col:<6} -> {mvar:<5} x {JETTY_FACTOR[mvar]}")
    for v in VALIDATION_VARS:
        if v not in out.columns:
            out[v] = np.nan
            print(f"       [INFO] no observed column for {v}; panel will be empty")

    out = add_derived(out)
    full_range = (out["timestamp"].min(), out["timestamp"].max())
    out = out[out["timestamp"].dt.year == MODEL_YEAR].sort_values("timestamp")
    out = out.reset_index(drop=True)
    print(f"       series covers {full_range[0].date()} – {full_range[1].date()}; "
          f"{len(out)} samples kept for {MODEL_YEAR}")
    return out

---
## 5. Model loading and collocation

The nutrient fields are 4-D `(time, level, yc, xc)`, so the surface layer is
selected **before** loading. Fill values (`-9999`, and the `-9998` that fills
the first frame of every file for averaged fields) become NaN, and at the
duplicated month boundaries the copy with valid data is kept.

Stations snap to the nearest wet cell by distance in km; the model series at
each station also carries the range over the surrounding
`(2*BOX_HALF+1)²` wet cells, drawn as a band — a measure of how much the
comparison depends on the exact cell at a 500 m coastal grid point.

In [ ]:
R_EARTH_KM = 6371.0


def load_grid(spinup_dir: Path) -> dict:
    """Static grid fields and the time convention of each variable."""
    first = next(iter(sorted(spinup_dir.glob("dws_500m.3d.*.nc"))), None)
    if first is None:
        raise FileNotFoundError(f"No NetCDF files in {spinup_dir}")
    with xr.open_dataset(first, mask_and_scale=False) as g:
        lon = g["lonc"].values.astype(float)
        lat = g["latc"].values.astype(float)
        bathy = g["bathymetry"].values.astype(float)
        averaged = {v: bool(np.ravel(g[v].attrs.get("averaged", [0]))[0])
                    for v in VALIDATION_VARS if v in g}
    valid = (lon > LON_VALID[0]) & (lon < LON_VALID[1]) & (lat > LAT_VALID[0]) & (lat < LAT_VALID[1])
    wet = valid & (bathy > BATHY_FILL)
    return {"lon": np.where(valid, lon, np.nan), "lat": np.where(valid, lat, np.nan),
            "depth": np.where(wet, bathy, np.nan), "valid": valid, "wet": wet,
            "averaged": averaged}


def time_convention(grid: dict) -> str:
    if MODEL_TIME_CONVENTION != "auto":
        return MODEL_TIME_CONVENTION
    flags = set(grid["averaged"].values())
    if len(flags) > 1:
        warnings.warn("some variables are averaged and some are not; using interval_mean")
    return "interval_mean" if True in flags else "instantaneous"


def xy_km(lon, lat, lat0):
    return np.column_stack([np.radians(lon) * np.cos(np.radians(lat0)) * R_EARTH_KM,
                            np.radians(lat) * R_EARTH_KM])


def snap_stations(stations: pd.DataFrame, grid: dict) -> pd.DataFrame:
    """Nearest wet cell (km) for every station; adds iy, ix, snap_km, model_depth_m."""
    wet = grid["wet"]
    lat0 = float(np.nanmean(grid["lat"][wet]))
    iy, ix = np.nonzero(wet)
    tree = cKDTree(xy_km(grid["lon"][wet], grid["lat"][wet], lat0))
    dist, nn = tree.query(xy_km(stations["lon"].to_numpy(), stations["lat"].to_numpy(), lat0))
    st = stations.copy()
    st["iy"], st["ix"], st["snap_km"] = iy[nn], ix[nn], dist
    st["model_depth_m"] = grid["depth"][st["iy"], st["ix"]]
    return st


def load_model_surface_year(spinup_dir: Path, year: int) -> xr.Dataset | None:
    """Surface layer of every monthly file of *year*, fill values masked, de-duplicated."""
    files = sorted(spinup_dir.glob(NC_PATTERN.format(year=year)))
    if not files:
        return None
    parts = []
    for fp in files:
        with xr.open_dataset(fp, mask_and_scale=False) as dsi:
            present = [v for v in VALIDATION_VARS if v in dsi.data_vars]
            missing = [v for v in VALIDATION_VARS if v not in dsi.data_vars]
            if missing:
                warnings.warn(f"{fp.name}: variables {missing} absent")
            if not present:
                continue
            sub = dsi[present]
            if "level" in sub.dims:
                sub = sub.isel(level=SURFACE_LEVEL)
            sub = sub.load()
        parts.append(sub.where(sub > DATA_FILL_THRESHOLD))
    if not parts:
        return None
    ds = xr.concat(parts, dim="time", data_vars="minimal", coords="minimal",
                   compat="override", join="override")
    score = sum(ds[v].notnull().sum(dim=[d for d in ds[v].dims if d != "time"]).values
                for v in ds.data_vars)
    t = ds["time"].values
    order = np.lexsort((-score, t))            # time ascending, most complete copy first
    keep = np.r_[True, t[order][1:] != t[order][:-1]]
    ds = ds.isel(time=order[keep])
    mt = pd.DatetimeIndex(ds["time"].values)
    print(f"  {len(files)} files -> {len(mt)} time steps ({mt[0]:%Y-%m-%d} .. {mt[-1]:%Y-%m-%d}), "
          f"median step {pd.Series(mt).diff().median()}, variables {list(ds.data_vars)}")
    return ds


def model_time_idx(model_times: pd.DatetimeIndex, obs_times, convention: str) -> np.ndarray:
    """Model time index paired with each observation.

    interval_mean: the value stamped t is the mean over the preceding interval,
    so a sample belongs to the first stamp at or after it. instantaneous: the
    nearest stamp.
    """
    mt = model_times.to_numpy(dtype="datetime64[ns]")
    ot = np.asarray(obs_times, dtype="datetime64[ns]")
    right = np.clip(np.searchsorted(mt, ot, side="left"), 0, len(mt) - 1)
    if convention == "interval_mean":
        return right
    left = np.clip(right - 1, 0, len(mt) - 1)
    return np.where(np.abs(mt[right] - ot) < np.abs(ot - mt[left]), right, left)


def station_series(ds: xr.Dataset, iy: int, ix: int, grid: dict) -> pd.DataFrame:
    """Model series at one cell, plus min / max over the surrounding wet cells."""
    h = BOX_HALF
    ny, nx = grid["wet"].shape
    y0, y1, x0, x1 = max(iy - h, 0), min(iy + h + 1, ny), max(ix - h, 0), min(ix + h + 1, nx)
    wet = xr.DataArray(grid["wet"][y0:y1, x0:x1], dims=("yc", "xc"))
    out = pd.DataFrame({"time": pd.to_datetime(ds["time"].values)})
    for v in VALIDATION_VARS:
        if v not in ds.data_vars:
            out[v] = out[f"{v}_lo"] = out[f"{v}_hi"] = np.nan
            continue
        out[v] = ds[v].isel(yc=iy, xc=ix).values
        block = ds[v].isel(yc=slice(y0, y1), xc=slice(x0, x1)).where(wet)
        out[f"{v}_lo"] = block.min(dim=("yc", "xc"), skipna=True).values
        out[f"{v}_hi"] = block.max(dim=("yc", "xc"), skipna=True).values
    out = add_derived(out)
    out["DIN_lo"] = out["N3n_lo"] + out["N4n_lo"]
    out["DIN_hi"] = out["N3n_hi"] + out["N4n_hi"]
    out["DIN_DIP_lo"] = out["DIN_lo"] / out["N1p_hi"].where(out["N1p_hi"] > 0.01)
    out["DIN_DIP_hi"] = out["DIN_hi"] / out["N1p_lo"].where(out["N1p_lo"] > 0.01)
    return out


def pair_samples(obs: pd.DataFrame, series: dict, convention: str) -> pd.DataFrame:
    """Model value for every observed sample, from the station's model series."""
    parts = []
    for st, g in obs.groupby("station"):
        if st not in series:
            continue
        ts = series[st]
        mt = pd.DatetimeIndex(ts["time"])
        it = model_time_idx(mt, g["timestamp"], convention)
        p = g.copy()
        p["model_time"] = mt[it]
        p["time_offset_h"] = (p["model_time"] - p["timestamp"]).dt.total_seconds() / 3600
        for v in ALL_VARS:
            p[f"{v}_model"] = ts[v].to_numpy()[it]
        parts.append(p)
    pairs = pd.concat(parts, ignore_index=True)
    stale = pairs["time_offset_h"].abs() > MAX_TIME_OFFSET_DAYS * 24
    if stale.any():
        print(f"  [WARN] {int(stale.sum())} of {len(pairs)} samples are more than "
              f"{MAX_TIME_OFFSET_DAYS:g} days from a model time step - model values voided")
        pairs.loc[stale, [f"{v}_model" for v in ALL_VARS]] = np.nan
    return pairs

---
## 6. Load, collocate, audit

Things worth reading in the printout: the time axis (≈365 daily steps), the
snap distances (below ~0.7 km, one cell diagonal), the number of samples voided
by the time guard (large = the run does not cover those months), and the unit
audit, where a median ratio near 4.43, 3.07 or 2.14 between observations and
model would betray an N/NO₃, P/PO₄ or Si/SiO₂ mass-basis mix-up.

In [ ]:
print("Loading observations ...")
jetty_obs = load_jetty()
rws_long = load_rws()
rws_obs = rws_to_wide(rws_long)
rws_year = rws_obs[rws_obs["timestamp"].dt.year == MODEL_YEAR].copy()
print(f"RWS: {len(rws_obs):,} samples in total, {len(rws_year):,} in {MODEL_YEAR} from "
      f"{rws_year['station'].nunique()} stations")

jetty_obs = jetty_obs.assign(station="NIOZ_JETTY", lon=JETTY_LON, lat=JETTY_LAT)
for v in VALIDATION_VARS:
    jetty_obs[f"{v}_below_lod"] = False
OBS = pd.concat([jetty_obs, rws_year], ignore_index=True)
OBS["source"] = np.where(OBS["station"] == "NIOZ_JETTY", "NIOZ Jetty", "RWS")
ANY_LOD = bool(OBS[[f"{v}_below_lod" for v in VALIDATION_VARS]].fillna(False).to_numpy().any())

stations = (OBS.groupby("station").agg(lon=("lon", "first"), lat=("lat", "first"),
                                       source=("source", "first"), n_samples=("timestamp", "size")))
order = [s for s in STATION_ORDER if s in stations.index]
order += sorted(set(stations.index) - set(order), key=lambda s: stations.loc[s, "lon"])
stations = stations.loc[order]
stations["label"] = [STATION_LABEL.get(s, s) for s in stations.index]

RESULTS = {}
for run in SPINUP_NAMES:
    run_dir = BASE_OUTPUT_DIR / run
    print(f"\n{run} ({run_dir})")
    if not run_dir.is_dir():
        print("  not found - skipped")
        continue
    grid = load_grid(run_dir)
    conv = time_convention(grid)
    print(f"  time convention: {conv}  (averaged flags: {grid['averaged']})")
    st = snap_stations(stations, grid)
    far = st["snap_km"] > MAX_SNAP_DISTANCE_KM
    if far.any():
        print(f"  [WARN] dropping stations > {MAX_SNAP_DISTANCE_KM} km from a wet cell: "
              f"{list(st.index[far])}")
        st = st[~far]
    display(st[["label", "source", "lon", "lat", "snap_km", "model_depth_m", "n_samples"]].round(3))
    ds = load_model_surface_year(run_dir, MODEL_YEAR)
    if ds is None:
        print(f"  no model files for {MODEL_YEAR} - skipped")
        continue
    series = {s: station_series(ds, int(r.iy), int(r.ix), grid) for s, r in st.iterrows()}
    ds.close()
    pairs = pair_samples(OBS[OBS["station"].isin(st.index)], series, conv)
    print(f"  paired samples: {len(pairs)}, median |time offset| "
          f"{pairs['time_offset_h'].abs().median():.1f} h")
    RESULTS[run] = {"grid": grid, "stations": st, "series": series, "pairs": pairs,
                    "convention": conv}

    out = OUT_DIR / run
    out.mkdir(parents=True, exist_ok=True)
    pd.concat({s: ts for s, ts in series.items()}, names=["station"]).reset_index(level=0).to_csv(
        out / f"model_surface_series_{MODEL_YEAR}.csv", index=False)
    pairs.to_csv(out / f"collocated_model_vs_obs_{MODEL_YEAR}.csv", index=False)
    st.to_csv(out / f"station_matching_{MODEL_YEAR}.csv")

assert RESULTS, "no model run could be collocated"
RUNS = list(RESULTS)
STATIONS = RESULTS[RUNS[0]]["stations"]

In [ ]:
def unit_audit(pairs: pd.DataFrame) -> pd.DataFrame:
    """Observed vs modelled percentiles per source; ratios flag mass-basis errors."""
    rows = []
    for v in ALL_VARS:
        lo, hi = PLAUSIBLE_RANGE.get(v, (np.nan, np.nan))
        for label, s in [("model", pairs.get(f"{v}_model"))] + [
                (f"obs {src}", pairs.loc[pairs["source"] == src, v]) for src in ("NIOZ Jetty", "RWS")]:
            s = pd.to_numeric(s, errors="coerce").dropna() if s is not None else pd.Series(dtype=float)
            rows.append({"variable": v, "source": label, "n": len(s),
                         "p5": s.quantile(0.05) if len(s) else np.nan,
                         "median": s.median() if len(s) else np.nan,
                         "p95": s.quantile(0.95) if len(s) else np.nan,
                         "plausible": f"{lo:g}-{hi:g}"})
    audit = pd.DataFrame(rows)
    med = audit.pivot_table(index="variable", columns="source", values="median", sort=False)
    for c in [c for c in med.columns if c.startswith("obs")]:
        med[f"{c} / model"] = med[c] / med["model"]
    display(med.round(3))
    return audit


def skill(o, m) -> dict:
    o, m = np.asarray(o, float), np.asarray(m, float)
    ok = np.isfinite(o) & np.isfinite(m)
    o, m = o[ok], m[ok]
    out = {"n": len(o), "obs_mean": np.nan, "mod_mean": np.nan, "bias": np.nan,
           "nbias_pct": np.nan, "rmse": np.nan, "r": np.nan}
    if len(o) < MIN_PAIRS:
        return out
    out.update(obs_mean=o.mean(), mod_mean=m.mean(), bias=(m - o).mean(),
               rmse=np.sqrt(((m - o) ** 2).mean()))
    out["nbias_pct"] = 100 * out["bias"] / o.mean() if o.mean() != 0 else np.nan
    if o.std() > 0 and m.std() > 0:
        out["r"] = pearsonr(o, m)[0]
    return out


METRICS = {}
for run, res in RESULTS.items():
    print(f"\n{run}: unit audit (medians; ratio ~4.43 / 3.07 / 2.14 = mass-basis error)")
    audit = unit_audit(res["pairs"])
    p = res["pairs"]
    rows = []
    for v in ALL_VARS:
        for st in list(res["stations"].index) + ["all RWS", "all"]:
            sel = (p["station"] == st if st not in ("all RWS", "all")
                   else (p["source"] == "RWS") if st == "all RWS" else np.ones(len(p), bool))
            rows.append({"variable": v, "station": st, **skill(p.loc[sel, v], p.loc[sel, f"{v}_model"])})
    METRICS[run] = pd.DataFrame(rows)
    audit.to_csv(OUT_DIR / run / f"unit_audit_{MODEL_YEAR}.csv", index=False)
    METRICS[run].round(4).to_csv(OUT_DIR / run / f"nutrient_metrics_{MODEL_YEAR}.csv", index=False)
    print("\nskill, all stations pooled:")
    display(METRICS[run][METRICS[run]["station"] == "all"].set_index("variable")
            [["n", "obs_mean", "mod_mean", "bias", "nbias_pct", "rmse", "r"]].round(3))

---
## 7. Figures

Model daily means are drawn at the middle of the day they average (12:00);
observations at their sampling time. Observations are ink; each model run has
its fixed colour, with a light band for the range over the surrounding wet
cells. Hollow markers are values below the detection limit (kept as LOD/2).

In [ ]:
def plot_times(ts: pd.DataFrame, run: str) -> pd.Series:
    """Model time stamps shifted to the middle of the interval they average."""
    t = pd.to_datetime(ts["time"])
    return t - pd.Timedelta(hours=12) if RESULTS[run]["convention"] == "interval_mean" else t


def month_axis(ax, labels=True):
    ax.set_xlim(pd.Timestamp(f"{MODEL_YEAR}-01-01"), pd.Timestamp(f"{MODEL_YEAR + 1}-01-01"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_minor_locator(mticker.NullLocator())
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: mdates.num2date(x).strftime("%b")[0] if labels else ""))


def obs_points(ax, t, v, lod, **kw):
    """Observed values: filled, or hollow where below the detection limit."""
    lod = np.asarray(lod, bool)
    base = dict(s=9, zorder=5, linewidths=0.6)
    base.update(kw)
    ax.scatter(np.asarray(t)[~lod], np.asarray(v)[~lod], color=fs.OBS_COLOUR, **base)
    if lod.any():
        ax.scatter(np.asarray(t)[lod], np.asarray(v)[lod], facecolor="white",
                   edgecolor=fs.OBS_COLOUR, **base)


def lod_flags(df, v):
    col = f"{v}_below_lod"
    return df[col].fillna(False).astype(bool).to_numpy() if col in df else np.zeros(len(df), bool)


def model_handles():
    return [Line2D([], [], color=RUN_COLOURS[r], lw=1.0, label=f"GETM {r}") for r in RUNS]

### Fig. 1 — stations

GSHHG full-resolution land, water shaded by model depth, and the computational
domain outlined in the run colour. Circles: NIOZ Jetty; squares: RWS stations.

In [ ]:
grid0 = RESULTS[RUNS[0]]["grid"]
LAND = fs.load_land(MAP_EXTENT)
LON_F = fs.fill_coord(grid0["lon"], verbose=False)
LAT_F = fs.fill_coord(grid0["lat"], verbose=False)
lat0 = 0.5 * (MAP_EXTENT[2] + MAP_EXTENT[3])
aspect = (MAP_EXTENT[1] - MAP_EXTENT[0]) * np.cos(np.radians(lat0)) / (MAP_EXTENT[3] - MAP_EXTENT[2])

fig, ax = fs.figure("single", (89 - 15) / aspect + 17)
fs.setup_map(ax, MAP_EXTENT, lat0=lat0, xstep=0.25, ystep=0.2)
fs.add_bathymetry(ax, LON_F, LAT_F, grid0["depth"])
fs.add_land(ax, LAND)
fs.add_domain_outline(ax, grid0["lon"], grid0["lat"], RUN_COLOURS[RUNS[0]], valid=grid0["valid"])
fs.add_water_label(ax, 4.68, 53.36, "North Sea")
fs.add_water_label(ax, 5.28, 53.12, "Wadden Sea")
fs.add_scalebar(ax, 10, "lower right")
for src in ("NIOZ Jetty", "RWS"):
    s = STATIONS[STATIONS["source"] == src]
    ax.scatter(s["lon"], s["lat"], marker=fs.SOURCE_MARKERS[src], s=18, facecolor="white",
               edgecolor=fs.INK, linewidth=0.8, zorder=8)
handles = [Line2D([], [], ls="", marker=fs.SOURCE_MARKERS[s], ms=4, mfc="white", mec=fs.INK,
                  mew=0.8, label=s) for s in ("NIOZ Jetty", "RWS")]
handles += [Line2D([], [], color=RUN_COLOURS[RUNS[0]], lw=0.7, label=f"model domain ({RUNS[0]})")]
fig.legend(handles=handles, loc="outside upper center", ncol=3)
fs.place_labels(ax, STATIONS["lon"], STATIONS["lat"], STATIONS["label"].tolist(), fontsize=6.5,
                path_effects=fs.halo())
fs.save_figure(fig, "fig01_nutrients_stations", FIG_DIR,
               data=STATIONS[["label", "source", "lon", "lat", "snap_km", "model_depth_m",
                              "n_samples"]])

### Fig. 2 — seasonal cycle at every station

One row per variable (shared y axis along the row, so the gradient from the
inlets to the North Sea is visible), one column per station from the Marsdiep
inlet to offshore. A panel without observations still shows the model.

In [ ]:
cols = list(STATIONS.index)
fig, axes = fs.figure("double", 20 + 27 * len(PLOT_VARS), nrows=len(PLOT_VARS), ncols=len(cols),
                      sharex=True, sharey="row", squeeze=False)
csv, no_obs = [], []
for i, v in enumerate(PLOT_VARS):
    for j, st in enumerate(cols):
        ax = axes[i, j]
        for run in RUNS:
            ts = RESULTS[run]["series"][st]
            t = plot_times(ts, run)
            ax.fill_between(t, ts[f"{v}_lo"], ts[f"{v}_hi"], color=RUN_COLOURS[run], alpha=0.18,
                            lw=0, zorder=1)
            ax.plot(t, ts[v], color=RUN_COLOURS[run], lw=0.9, zorder=3)
            csv.append(pd.DataFrame({"run": run, "station": st, "variable": v, "kind": "model",
                                     "time": t, "value": ts[v], "lo": ts[f"{v}_lo"],
                                     "hi": ts[f"{v}_hi"]}))
        o = RESULTS[RUNS[0]]["pairs"]
        o = o[(o["station"] == st) & o[v].notna()]
        if len(o):
            obs_points(ax, o["timestamp"], o[v], lod_flags(o, v))
            csv.append(pd.DataFrame({"run": "", "station": st, "variable": v, "kind": "observed",
                                     "time": o["timestamp"], "value": o[v], "lo": np.nan,
                                     "hi": np.nan}))
        else:
            ts0 = RESULTS[RUNS[0]]["series"][st]
            no_obs.append((ax, mdates.date2num(plot_times(ts0, RUNS[0])), ts0[v].to_numpy()))
        month_axis(ax, i == len(PLOT_VARS) - 1)
        ax.yaxis.set_major_locator(mticker.MaxNLocator(4))
        if i == 0:
            ax.set_title(STATIONS.loc[st, "label"])
        if j == 0:
            ax.set_ylabel(f"{VAR_LABEL[v]}\n({VAR_UNITS[v]})")
    if v != "O2o":                       # concentrations start at zero; oxygen does not
        axes[i, 0].set_ylim(bottom=0)
handles = model_handles()
handles += [plt.Rectangle((0, 0), 1, 1, color=RUN_COLOURS[RUNS[0]], alpha=0.18, lw=0,
                          label=f"range over {2 * BOX_HALF + 1}×{2 * BOX_HALF + 1} cells"),
            Line2D([], [], ls="", marker="o", ms=3, color=fs.OBS_COLOUR, label="observed")]
if ANY_LOD:
    handles.append(Line2D([], [], ls="", marker="o", ms=3, mfc="white", mec=fs.OBS_COLOUR,
                          label="below detection limit"))
fig.legend(handles=handles, loc="outside upper center", ncol=len(handles))
for ax, l in zip(axes.flat, "abcdefghijklmnopqrstuvwxyz" + "ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
    fs.panel_label(ax, l)
for ax, x, y in no_obs:                  # after the layout is final, so corners are right
    fs.corner_note(ax, "no observations", x, y, box=(0.5, 0.22), fontsize=6,
                   style="italic", color=fs.INK3,
                   corners=("upper right", "upper left", "lower right", "lower left"))
fs.save_figure(fig, "fig02_nutrients_timeseries", FIG_DIR, data=pd.concat(csv, ignore_index=True))

### Fig. 3 — model vs observed, all stations pooled

Circles: NIOZ Jetty; squares: RWS stations; hollow (if any): below the detection
limit.
Dashed line 1:1. Scores above each panel use all pairs together.

In [ ]:
have = [v for v in PLOT_VARS if any(RESULTS[r]["pairs"][[v, f"{v}_model"]].notna().all(axis=1).sum()
                                    >= MIN_PAIRS for r in RUNS)]
ncol = 3
nrow = int(np.ceil(len(have) / ncol))
fig, axes = fs.figure("double", 14 + 60 * nrow, nrows=nrow, ncols=ncol, squeeze=False)
csv = []
for k, v in enumerate(have):
    ax = axes[k // ncol, k % ncol]
    lines, hi, lo = [], 0.0, np.inf
    for run in RUNS:
        p = RESULTS[run]["pairs"]
        p = p[p[v].notna() & p[f"{v}_model"].notna()]
        hi = max(hi, float(np.nanmax(p[[v, f"{v}_model"]].to_numpy())))
        lo = min(lo, float(np.nanmin(p[[v, f"{v}_model"]].to_numpy())))
        for src in ("NIOZ Jetty", "RWS"):
            q = p[p["source"] == src]
            if q.empty:
                continue
            lod = lod_flags(q, v)
            kw = dict(marker=fs.SOURCE_MARKERS[src], s=12, linewidths=0.6, zorder=3)
            ax.scatter(q[v][~lod], q[f"{v}_model"][~lod], color=RUN_COLOURS[run], alpha=0.8, **kw)
            if lod.any():
                ax.scatter(q[v][lod], q[f"{v}_model"][lod], facecolor="white",
                           edgecolor=RUN_COLOURS[run], **kw)
            csv.append(pd.DataFrame({"run": run, "variable": v, "source": src,
                                     "station": q["station"], "time": q["timestamp"],
                                     "observed": q[v], "modelled": q[f"{v}_model"],
                                     "below_lod": lod}))
        s = skill(p[v], p[f"{v}_model"])
        lines.append((f"{run}: " if len(RUNS) > 1 else "")
                     + f"n {s['n']} · bias {fs.signed(s['bias'], '.3g')} · "
                       f"RMSE {s['rmse']:.3g} · r {fs.unsigned(s['r'])}")
    # start at zero unless the data sit far above it (oxygen)
    lo = 0.0 if lo < 0.4 * hi else lo - 0.06 * (hi - lo)
    hi += 0.06 * (hi - lo)
    ax.plot([lo, hi], [lo, hi], ls=(0, (4, 2)), lw=0.6, color=fs.INK, zorder=1)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_box_aspect(1)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(4))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(4))
    ax.set_title(VAR_LABEL[v], pad=4 + 8 * len(lines))
    fs.stats_line(ax, "\n".join(lines))
    ax.set_xlabel(f"Observed ({VAR_UNITS[v]})")
    ax.set_ylabel(f"Modelled ({VAR_UNITS[v]})")
    fs.panel_label(ax, "abcdefghij"[k])
for k in range(len(have), nrow * ncol):
    axes[k // ncol, k % ncol].axis("off")
handles = [Line2D([], [], ls="", marker=fs.SOURCE_MARKERS[s], ms=3.6, color=RUN_COLOURS[RUNS[0]],
                  label=s) for s in ("NIOZ Jetty", "RWS")]
if ANY_LOD:
    handles.append(Line2D([], [], ls="", marker="o", ms=3.6, mfc="white",
                          mec=RUN_COLOURS[RUNS[0]], label="below detection limit"))
handles.append(Line2D([], [], ls=(0, (4, 2)), lw=0.6, color=fs.INK, label="1:1"))
fig.legend(handles=handles, loc="outside upper center", ncol=len(handles))
fs.save_figure(fig, "fig03_nutrients_scatter", FIG_DIR, data=pd.concat(csv, ignore_index=True))

### Fig. 4 — scorecard

Colour: normalised bias, (model − observed) / observed mean, on a symmetric
scale (±100 %, arrows beyond). Text: Pearson *r* and the number of pairs. Grey:
no (or fewer than `MIN_PAIRS`) observations. With so few samples per station
(12–40), treat *r* as indicative.

In [ ]:
SCORE_VARS = PLOT_VARS + ["DIN", "DIN_DIP"]
for run in RUNS:
    m = METRICS[run]
    cols = list(RESULTS[run]["stations"].index) + ["all"]
    col_lab = [STATIONS.loc[c, "label"] if c in STATIONS.index else "All stations" for c in cols]
    nb = m.pivot_table(index="variable", columns="station", values="nbias_pct", dropna=False
                       ).reindex(index=SCORE_VARS, columns=cols)
    rr = m.pivot_table(index="variable", columns="station", values="r", dropna=False
                       ).reindex(index=SCORE_VARS, columns=cols)
    nn = m.pivot_table(index="variable", columns="station", values="n", dropna=False
                       ).reindex(index=SCORE_VARS, columns=cols).fillna(0)
    lim = 100.0
    norm = mcolors.Normalize(-lim, lim)
    fig, ax = fs.figure("onehalf", 16 + 7.5 * len(SCORE_VARS))
    fs.open_frame(ax)
    cmap = fs.cmap_diverging().copy()
    cmap.set_bad("#efeee9")
    arr = np.ma.masked_invalid(np.where(nn.to_numpy() >= MIN_PAIRS, nb.to_numpy(), np.nan))
    im = ax.pcolormesh(np.arange(len(cols) + 1), np.arange(len(SCORE_VARS) + 1), arr, cmap=cmap,
                       norm=norm, edgecolors="white", linewidth=1.0)
    for i in range(len(SCORE_VARS)):
        for j in range(len(cols)):
            if nn.iat[i, j] < MIN_PAIRS:
                ax.text(j + 0.5, i + 0.5, "–", ha="center", va="center", fontsize=6.5,
                        color=fs.INK3)
                continue
            val = arr[i, j]
            dark = abs(val) > 60 if np.isfinite(val) else False
            r_txt = f"r {fs.unsigned(rr.iat[i, j])}" if np.isfinite(rr.iat[i, j]) else "r –"
            ax.text(j + 0.5, i + 0.5, f"{r_txt}\nn {int(nn.iat[i, j])}", ha="center",
                    va="center", fontsize=5.8, linespacing=1.15,
                    color="white" if dark else fs.INK)
    ax.set_xticks(np.arange(len(cols)) + 0.5, col_lab, rotation=0)
    ax.set_yticks(np.arange(len(SCORE_VARS)) + 0.5, [VAR_LABEL[v] for v in SCORE_VARS])
    ax.tick_params(which="both", length=0)
    ax.minorticks_off()
    ax.set_ylim(len(SCORE_VARS), 0)
    ax.xaxis.tick_top()
    fs.diverging_colorbar(fig, im, ax=ax, extend="both", ticks=[-100, -50, 0, 50, 100],
                          label="Normalised bias (%)", shrink=0.8, aspect=20, pad=0.03)
    fs.save_figure(fig, f"fig04_nutrients_scorecard_{run}", FIG_DIR,
                   data=m[m["variable"].isin(SCORE_VARS)])

### Fig. 5 — nutrient stoichiometry

DIN:DIP on a log axis, with the Redfield ratio (16 mol N : 1 mol P) dashed.
Above it phosphorus is scarce relative to nitrogen, below it nitrogen. Ratios
are only formed where phosphate exceeds 0.01 mmol m⁻³, so summer values that
would divide by almost nothing are left out.

In [ ]:
v = "DIN_DIP"
cols = list(STATIONS.index)
fig, axes = fs.figure("double", 52, ncols=len(cols), sharey=True, squeeze=False)
csv = []
for j, st in enumerate(cols):
    ax = axes[0, j]
    ax.axhline(REDFIELD_NP, color=fs.INK, lw=0.6, ls=(0, (4, 2)), zorder=1)
    for run in RUNS:
        ts = RESULTS[run]["series"][st]
        t = plot_times(ts, run)
        ax.plot(t, ts[v], color=RUN_COLOURS[run], lw=0.9, zorder=3)
        csv.append(pd.DataFrame({"run": run, "station": st, "kind": "model", "time": t,
                                 "DIN_DIP": ts[v]}))
    o = RESULTS[RUNS[0]]["pairs"]
    o = o[(o["station"] == st) & o[v].notna()]
    if len(o):
        obs_points(ax, o["timestamp"], o[v], np.zeros(len(o), bool))
        csv.append(pd.DataFrame({"run": "", "station": st, "kind": "observed",
                                 "time": o["timestamp"], "DIN_DIP": o[v]}))
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:g}"))
    month_axis(ax)
    ax.set_title(STATIONS.loc[st, "label"])
    if j == 0:
        ax.set_ylabel(f"DIN:DIP ({VAR_UNITS[v]})")
    fs.panel_label(ax, "abcdefgh"[j])
axes[0, 0].annotate("Redfield 16:1", xy=(0.02, REDFIELD_NP), xycoords=("axes fraction", "data"),
                    xytext=(0, -2), textcoords="offset points", ha="left", va="top",
                    fontsize=6, color=fs.INK2)
handles = model_handles() + [Line2D([], [], ls="", marker="o", ms=3, color=fs.OBS_COLOUR,
                                    label="observed")]
fig.legend(handles=handles, loc="outside upper center", ncol=len(handles))
fs.save_figure(fig, "fig05_nutrients_stoichiometry", FIG_DIR, data=pd.concat(csv, ignore_index=True))

---
## 8. Output index

In [ ]:
print("written to", FIG_DIR)
for f in sorted(FIG_DIR.glob("fig*")):
    print(f"   {f.name:<44} {f.stat().st_size / 1024:>8.0f} kB")